# Conversions: int, bytes, str

> Execute the example cells in this introductory part; they are just demonstrations. The actual tasks begin below the horizontal rule.

| from \ to | `int` | `bytes` | `str` |
|---|---|---|---|
| `int` | – | `x.to_bytes(n, 'big')` | `hex(x)`<br>`bin(x)`<br>`str(x)` |
| `bytes` | `int.from_bytes(b, 'big')` | – | `b.hex()`<br>`b.decode('utf-8')`<br>`base64.b64encode(b).decode()` |
| `str` | `int(s, base)` | `bytes.fromhex(s)`<br>`s.encode('utf-8')`<br>`base64.b64decode(s)` | – |

## 1. Defining values

In [1]:
print("int:", 0b101, 0o101, 101, 0x101)  # same digits in base 2 (0b), 8 (0o), 10, 16 (0x)
print("str:", "101", "0b101", "0o101", "0x101", "00ff", "SGVsbG8=", "Hello")  # text: decimal, binary, octal, hex, hex bytes, base64 of "Hello", plain text
print("bytes:", b"101", b"\x31\x30\x31", bytes([49, 48, 49]), b"\x00\xff")  # ASCII chars, hex escapes \xHH, ints 0-255
ba = bytearray(b"101"); ba[0] = 0x32
print("bytearray:", ba)

int: 5 65 101 257
str: 101 0b101 0o101 0x101 00ff SGVsbG8= Hello
bytes: b'101' b'101' b'101' b'\x00\xff'
bytearray: bytearray(b'201')


**Pitfalls**
- `b"101"` are the ASCII characters `1 0 1` (bytes `0x31 0x30 0x31`), not the number 101 – Python prints bytes as ASCII where possible.
- One element of bytes is an `int`: `b"101"[0] == 49`. Each value must be 0–255, `bytes([256])` raises `ValueError`.
- `bytes` cannot be changed (`TypeError`); use `bytearray` if you need to.

## 2. int ↔ str (binary, octal, decimal, hex)

In [2]:
# str -> int: int(s, base); base 0 detects the base from the prefix 0b / 0o / 0x
print("base 2: ", int('1011', 2), int('0b1011', 2))
print("base 8: ", int('1011', 8), int('0o1011', 8))
print("base 10:", int('1011'), int('1011', 10))
print("base 16:", int('ff', 16), int('0xFF', 16), int('0xff', 0))

# int -> str
print("with prefix:", bin(255), oct(255), hex(255), str(255))
print("format:     ", format(255, 'b'), format(255, 'o'), format(255, 'x'), format(255, 'X'))
print("fixed width:", format(5, '08b'), format(10, '02x'), f"{5:08b} {10:02x} {255:#x}")

base 2:  11 11
base 8:  521 521
base 10: 1011 1011
base 16: 255 255 255
with prefix: 0b11111111 0o377 0xff 255
format:      11111111 377 ff FF
fixed width: 00000101 0a 00000101 0a 0xff


**Pitfalls**
- `bin()` and `hex()` add the prefix `0b` / `0x`; `format(x, 'b')`, `format(x, 'x')` do not.
- Leading zeros are lost: `bin(1)` is `'0b1'`. Give a width: `format(1, '08b')`.
- Without a base `int()` expects a decimal string: `int('ff')` raises `ValueError`, use `int('ff', 16)`.

## 3. int ↔ bytes

In [3]:
print("bytes to int:", int.from_bytes(b'\x01\x02\x03\x04', 'big'), int.from_bytes(b'\x01\x02\x03\x04', 'little'))
print("int to bytes:", (16909060).to_bytes(4, 'big'), (16909060).to_bytes(4, 'little'))
x = 0x0102
print("minimal length:", x.to_bytes((x.bit_length() + 7) // 8, 'big'))

bytes to int: 16909060 67305985
int to bytes: b'\x01\x02\x03\x04' b'\x04\x03\x02\x01'
minimal length: b'\x01\x02'


### Little vs big endian
- **Little endian**: the index of a byte is the exponent of its weight, $B_0 + B_1 \cdot 256 + B_2 \cdot 256^2 + \dots$ – LSB first. This is how CPUs (x86, ARM) store integers: a C program reads the bytes `01 02 03 04` as `0x04030201`.
- **Big endian**: MSB first, as we write numbers on paper. Used in network protocols and ASN.1/DER.

Printed left to right, a little-endian number looks "reversed".

In [4]:
def my_bytes_to_littlee(byte_array):
    res = 0
    for i in range(len(byte_array)):
        res += byte_array[i]*256**i
    return res

def my_bytes_to_int(byte_array):
    res = 0
    for i in range(len(byte_array)):
        res = res*256 + byte_array[i]
    return res

byte_array = [1, 2, 3, 4]
print(my_bytes_to_littlee(byte_array), int.from_bytes(byte_array, 'little'))
print(my_bytes_to_int(byte_array), int.from_bytes(byte_array, 'big'))

67305985 67305985
16909060 16909060


**Pitfalls**
- The same bytes give different numbers in big and little endian – always know which one you need.
- `to_bytes` needs a length; too small raises `OverflowError`. Negative numbers need `signed=True`.
- `67305985.to_bytes(4, 'big')` is a `SyntaxError` (`67305985.` is a float literal) – write `(67305985).to_bytes(4, 'big')`.
- A leading zero byte is lost in the round trip bytes → int → bytes unless you keep the original length:

In [5]:
b = b'\x00\x01'
print(int.from_bytes(b, 'big').to_bytes(1, 'big'), int.from_bytes(b, 'big').to_bytes(len(b), 'big'))

b'\x01' b'\x00\x01'


## 4. bytes ↔ str (hex, text)

In [6]:
print("hex:", bytes.fromhex('00ff'), bytes([0, 255]).hex())
print("text to bytes:", 'Aa1'.encode('ascii'), '₹'.encode('utf-8'))
print("bytes to text:", b'Aa1'.decode('ascii'), b'\xe2\x82\xb9'.decode('utf-8'))

hex: b'\x00\xff' 00ff
text to bytes: b'Aa1' b'\xe2\x82\xb9'
bytes to text: Aa1 ₹


**Pitfalls**
- A hex string needs two digits per byte: `bytes.fromhex('fff')` raises `ValueError`.
- `str(b'Aa1')` does not decode, it gives `"b'Aa1'"` – use `.decode()` or `.hex()`.
- Not every character fits into ASCII (`'₹'.encode('ascii')` → `UnicodeEncodeError`) and not all bytes are valid UTF-8 (`b'\xff'.decode('utf-8')` → `UnicodeDecodeError`).
- One character can be several bytes: `len('₹') == 1`, but `len('₹'.encode('utf-8')) == 3`.

## 5. Base64
Every 3 bytes become 4 characters `A-Za-z0-9+/`, padded with 0–2 `=`. It is the text inside PEM files.

In [7]:
import base64
print("bytes to base64:", base64.b64encode(b'\x01\x02'))
print("base64 to bytes:", base64.b64decode(b'AQI='))
print("padding:", base64.b64encode(b'\x00'), base64.b64encode(b'\x00\x01'), base64.b64encode(b'\x00\x02\x03'))

bytes to base64: b'AQI='
base64 to bytes: b'\x01\x02'
padding: b'AA==' b'AAE=' b'AAID'


**Pitfalls**
- `b64encode` returns **bytes**, not `str` – use `.decode('ascii')`.
- Missing padding makes decoding fail: `base64.b64decode('AQI')` raises `binascii.Error`.
- Base64 is an encoding, not encryption – anybody can decode it.

---
> The cells above are explanatory examples; execute them as needed. The actual tasks start here.

# Tasks

**Task 1**: Test correctness of implementation of AES - use following test vectors:
```
pt  = 00112233445566778899aabbccddeeff
key = 000102030405060708090a0b0c0d0e0f
ct  = 69c4e0d86a7b0430d8cdb78070b4c55a
```
More test vectors can be found here: [AES128 test vectors](https://nvlpubs.nist.gov/nistpubs/fips/nist.fips.197.pdf). 

In [8]:
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes

pt  = bytes.fromhex('00112233445566778899aabbccddeeff')
key = bytes.fromhex('000102030405060708090a0b0c0d0e0f')
ct  = bytes.fromhex('69c4e0d86a7b0430d8cdb78070b4c55a')

encryptor = Cipher(algorithms.AES(key), modes.ECB()).encryptor()
result = encryptor.update(pt) + encryptor.finalize()
assert result == ct
print(result.hex())

69c4e0d86a7b0430d8cdb78070b4c55a


**Task 2**: Verify that $$y^2=x^3+ax+b \mod p$$  holds for point G = (x, y). First transform strings to byte arrays and then to integers! 

EC taken from not official source [EC](https://www.analog.com/en/technical-articles/elliptic-curve-digital-signature-algorithm-explained.html) but proper source can be found here [SEC 2](https://www.secg.org/sec2-v2.pdf) – the curve is P-192 (secp192r1).

In [9]:
p = "FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFFF"
a = 'FFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEFFFFFFFFFFFFFFFC'
b = '64210519E59C80E70FA7E9AB72243049FEB8DEECC146B9B1'
G = ('188DA80EB03090F67CBF20EB43A18800F4FF0AFD82FF1012', '07192B95FFC8DA78631011ED6B24CDD573F977A11E794811')
p = int.from_bytes(bytes.fromhex(p), byteorder='big')
a = int.from_bytes(bytes.fromhex(a), byteorder='big')
b = int.from_bytes(bytes.fromhex(b), byteorder='big')
x = int.from_bytes(bytes.fromhex(G[0]), byteorder='big')
y = int.from_bytes(bytes.fromhex(G[1]), byteorder='big')
lhs = (y*y) % p
rhs = (x*x*x + a*x + b) % p
assert lhs == rhs

**Task 3**: Generate random bytes with OpenSSL and look at them in different representations.
- Why does `-base64 16` give 24 characters?
- Which representation of `rnd.bin` can have a different length for another 10 random bytes?

In [10]:
!openssl rand -hex 16
!openssl rand -base64 16
!openssl rand -out rnd.bin 10

ddc7a6401d0adecb35fda342c4849696


Cyq/NrFA+A2MlqYLY75myw==


In [11]:
import base64
rnd = open("rnd.bin", "rb").read()
print(rnd.hex(), int.from_bytes(rnd, "big"), base64.b64encode(rnd))

b7b66174c744a96ab8bb 867557396243376903993531 b't7ZhdMdEqWq4uw=='


### Answers
Check your conclusion against the solution notebook and write it in the code cell below for comparison.

- Base64 length: every 3 bytes become 4 characters; 16 bytes = 5 × 3 + 1 → 5 × 4 + 4 characters (the last group padded with `==`) = 24.
- Variable-length representation: the integer – leading zero bytes do not count, so the number of digits changes. Hex (`.hex()`) always has 2 characters per byte and Base64 of 10 bytes always has 16 characters.

In [ ]:
# Final answer for comparison:
# base64_length = 24
# variable_length_representation = "int"  # integer form may vary in length